# B3 - Classical models

Two gradient-boosting models on the full feature bundle:

* **CatBoostQuantileForecaster** - one booster per quantile.
* **LightGBMQuantileForecaster** - one booster per quantile.

Evaluation uses 3-fold rolling-origin CV inside the train split, with a
24-hour gap between train and val windows to prevent lookahead leakage
through lag features that span fold boundaries.

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().resolve()
while not (REPO_ROOT / 'src').exists() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT / 'src'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 180)

## Load splits and build the full feature bundle

In [ ]:
from data.loaders import load_split
from data.splits import rolling_origin_folds
from module_b.features import prepare_supervised, HORIZON_COL
from module_b.features import build_features

train_df = load_split('train')
feat = build_features(train_df, bundles=(
    'calendar', 'lags', 'fundamentals', 'spike', 'regime', 'weather'))

past_cols = [c for c in feat.columns if
             c.startswith('price_lag') or c.startswith('price_rmean')
             or c.startswith('price_rstd') or c in ('residual_load',
                                                    'renewable_penetration',
                                                    'clean_spark_anchor',
                                                    'clean_dark_anchor',
                                                    'is_high_residual_load',
                                                    'is_renewable_scarcity',
                                                    'crisis_2022_flag')]
future_cols = ['hour_sin', 'hour_cos', 'dow_sin', 'dow_cos',
               'is_weekend', 'is_holiday_DE',
               'weather_mean_wind_speed_10m', 'weather_mean_wind_speed_100m',
               'weather_mean_shortwave_radiation', 'weather_mean_temperature_2m']
future_cols = [c for c in future_cols if c in feat.columns]
X, y = prepare_supervised(feat, horizons=range(1, 25),
                          past_cols=past_cols, future_cols=future_cols)
print(f'X.shape={X.shape}, n_features={len([c for c in X.columns if c not in ("origin_ts","target_ts")])}')

## 3-fold rolling-origin CV

Each fold has a 2-year train window followed by a 90-day val window,
with a 24h gap. Folds slide forward by `val_size` per step.

In [ ]:
folds = list(rolling_origin_folds(
    train_df.index, n_folds=5,
    train_size=pd.Timedelta(days=730),
    val_size=pd.Timedelta(days=90),
))
for f in folds:
    print(f'fold {f.fold_id}: train {f.train_start.date()} → {f.train_end.date()}  '
          f'val {f.val_start.date()} → {f.val_end.date()}')

## Define the model factories

Light hyperparameters so the whole CV finishes in minutes. For a full
comparison increase `iterations`/`num_boost_round` and add `X_val`/`y_val`
for early stopping.

In [ ]:
from module_b.models import (
    CatBoostQuantileForecaster, LightGBMQuantileForecaster,
)


## Run CV - one cell per model

Each cell below trains a single model across all folds and writes its results to
`outputs/B3_cv_<model>.parquet` *after every fold*. If one model hangs, you can interrupt that cell and the other models'
results stay safe on disk. After every model you care about has finished, run the
**Aggregate** cell to load every partial parquet and rebuild `results`.


In [ ]:
from module_b.evaluation import mae, multi_pinball_loss, coverage
from module_b.features import HORIZON_RANGES, HorizonGroup

outputs_dir = REPO_ROOT / 'notebooks' / 'module_b' / 'outputs'
outputs_dir.mkdir(exist_ok=True)

def run_model_cv(name, factory):
    """Train one model across all folds; save partial parquet after each fold."""
    out_path = outputs_dir / f'B3_cv_{name}.parquet'
    rows = []
    for f in folds:
        tr_mask = (X['origin_ts'] >= f.train_start) & (X['origin_ts'] <= f.train_end)
        va_mask = (X['origin_ts'] >= f.val_start) & (X['origin_ts'] <= f.val_end)
        m = factory()
        m.fit(X.loc[tr_mask], y.loc[tr_mask],
              X_val=X.loc[va_mask], y_val=y.loc[va_mask])
        q = m.predict_quantiles(X.loc[va_mask])
        y_va = y.loc[va_mask].to_numpy()
        for group in HorizonGroup:
            mask = X.loc[va_mask, HORIZON_COL].isin(HORIZON_RANGES[group]).to_numpy()
            if not mask.any():
                continue
            rows.append({
                'model': name, 'fold': f.fold_id,
                'horizon_group': group.value,
                'mae': mae(y_va[mask], q['q50'].to_numpy()[mask]),
                'pinball_avg': multi_pinball_loss(y_va[mask], q.iloc[mask], [0.1, 0.5, 0.9]),
                'coverage_80': coverage(y_va[mask], q['q10'].to_numpy()[mask], q['q90'].to_numpy()[mask]),
            })
        pd.DataFrame(rows).to_parquet(out_path)
        print(f'  \u2713 {name} fold {f.fold_id}  \u2192  {out_path.name}')
    return pd.DataFrame(rows)


### CatBoost

In [ ]:
_ = run_model_cv(
    'catboost_q',
    lambda: CatBoostQuantileForecaster(iterations=400, early_stopping_rounds=30),
)


### LightGBM

In [ ]:
_ = run_model_cv(
    'lightgbm_q',
    lambda: LightGBMQuantileForecaster(num_boost_round=400, early_stopping_rounds=30),
)


## Full training - class defaults, no light caps

These cells call each factory **without overriding the boost-round / iteration
caps**, so they run with the full defaults from `src/module_b/models.py`:

* CatBoost: `iterations=5000`, `early_stopping_rounds=100`
* LightGBM: `num_boost_round=2000`, `early_stopping_rounds=100`

Each cell saves its CV rows to `outputs/B3_cv_<model>_full.parquet` (distinct
from the light parquets above), so light + full results coexist.

**Expected wall-time on M3 Pro (3 folds \u00d7 3 quantiles):**

* CatBoost full \u2248 30\u201360 min
* LightGBM full \u2248 30\u201360 min


### CatBoost - full

In [ ]:
_ = run_model_cv(
    'catboost_q_full',
    lambda: CatBoostQuantileForecaster(learning_rate=0.02, depth=8),
)


### LightGBM - full

In [ ]:
_ = run_model_cv('lightgbm_q_full', lambda: LightGBMQuantileForecaster())


## Aggregate - rebuild `results` from saved partial parquets

Run this after every model you want to include is done. It just concats whatever
`B3_cv_*.parquet` files are present, so partial runs are still useful.


In [ ]:
import glob
parts = sorted(outputs_dir.glob('B3_cv_*.parquet'))
results = pd.concat([pd.read_parquet(p) for p in parts], ignore_index=True)
print(f'Loaded {len(parts)} partial parquets, {len(results)} rows total')
results.head()


## Leaderboard - mean ± std over folds

In [ ]:
agg = (results
       .groupby(['model', 'horizon_group'])[['mae', 'pinball_avg', 'coverage_80']]
       .agg(['mean', 'std']))
agg.round(3)

## Compare against baselines (B2)

In [ ]:
baselines = pd.read_parquet(REPO_ROOT / 'notebooks' / 'module_b' / 'outputs' / 'B2_baseline_results.parquet')
baselines['fold'] = -1  # not CV - single train/val
combined = pd.concat([baselines, results], ignore_index=True)
(combined.groupby(['model', 'horizon_group'])['pinball_avg']
 .mean().unstack().round(3).sort_values('h1_6'))

## Feature importance (CatBoost on a single fold)

In [ ]:
tr_mask = (X['origin_ts'] >= folds[0].train_start) & (X['origin_ts'] <= folds[0].train_end)
cb = CatBoostQuantileForecaster(iterations=400, early_stopping_rounds=30)
cb.fit(X.loc[tr_mask], y.loc[tr_mask])
imp = pd.Series(cb._models[0.5].get_feature_importance(),
                index=cb._feature_names).sort_values(ascending=False).head(15)
imp.plot.barh(figsize=(7, 6))
plt.title('CatBoost top-15 features (fold 0)')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

## Save results for B5/B6

In [ ]:
outputs_dir = REPO_ROOT / 'notebooks' / 'module_b' / 'outputs'
outputs_dir.mkdir(exist_ok=True)
results.to_parquet(outputs_dir / 'B3_classical_results.parquet')
print(f'Wrote {outputs_dir / "B3_classical_results.parquet"}')